# Module 4: Commodity Data Sources and Quality

## Learning Objectives

By the end of this module, you will:
1. Understand the landscape of commodity data sources (free and premium)
2. Access data programmatically via APIs (yfinance, FRED, EIA)
3. Assess data quality across multiple dimensions
4. Handle common data issues while maintaining temporal discipline
5. Build a robust data pipeline for commodity research

## Why Data Quality Matters

**"Garbage in, garbage out"** is especially true in commodity trading:
- Bad data leads to false signals
- Missing data creates look-ahead bias
- Delayed data makes backtests unrealistic
- Inconsistent data breaks models in production

This module teaches you to source, validate, and clean data properly.

In [ ]:
# Setup and imports
import sys
sys.path.append('../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Fair Value Toolkit imports
from src.fair_value_toolkit import (
    PointInTimeDataFrame,
)

from data.data_fetchers import (
    CommodityDataFetcher,
    COMMODITY_TICKERS,
    FRED_SERIES,
    create_sample_dataset,
    prepare_training_data,
)

# Plotting configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("Environment ready!")

## Part 1: Data Source Overview

### 1.1 Free Data Sources

#### Yahoo Finance
- **Coverage**: Futures prices for major commodities
- **Frequency**: Daily (historical), real-time (delayed)
- **History**: Typically 5-10 years
- **Cost**: Free
- **Reliability**: Good for price data, limited fundamentals

#### FRED (Federal Reserve Economic Data)
- **Coverage**: Economic indicators, some commodity spot prices, inventories
- **Frequency**: Various (daily, weekly, monthly)
- **History**: Extensive (decades for many series)
- **Cost**: Free
- **Reliability**: Excellent, official government data

#### EIA (Energy Information Administration)
- **Coverage**: Energy commodities (oil, gas, coal, electricity)
- **Frequency**: Various (weekly inventories, monthly production)
- **History**: Extensive
- **Cost**: Free
- **Reliability**: Excellent, official government data

In [ ]:
# Display available commodity tickers
print("Available Commodity Tickers (Yahoo Finance):\n")
for category in ['Energy', 'Precious Metals', 'Base Metals', 'Agriculture']:
    print(f"\n{category}:")
    
tickers_df = pd.DataFrame.from_dict(COMMODITY_TICKERS, orient='index', columns=['Ticker'])
print(tickers_df.to_string())

In [ ]:
# Display available FRED series
print("Available FRED Series:\n")
fred_df = pd.DataFrame.from_dict(FRED_SERIES, orient='index', columns=['Series Code'])
print(fred_df.to_string())

### 1.2 Premium Data Sources (Overview)

While we'll focus on free sources in this course, be aware of premium options:

#### Bloomberg Terminal
- **Coverage**: Comprehensive (prices, fundamentals, news, analytics)
- **Cost**: ~$24,000/year per user
- **Best for**: Professional traders, institutional research

#### Refinitiv (formerly Thomson Reuters)
- **Coverage**: Similar to Bloomberg
- **Cost**: Similar to Bloomberg
- **Best for**: Institutional users

#### Quandl (now part of Nasdaq)
- **Coverage**: Curated alternative data, commodities
- **Cost**: Varies ($50-$1000+/month)
- **Best for**: Quant researchers, algo traders

#### CME Group DataMine
- **Coverage**: Detailed futures/options data
- **Cost**: Varies by dataset
- **Best for**: Futures-focused strategies

#### IEA, OPEC, USDA Reports
- **Coverage**: Industry-specific fundamentals
- **Cost**: Free to moderate
- **Best for**: Fundamental analysis

### 1.3 Data Characteristics by Source

In [ ]:
# Create comparison table
comparison = pd.DataFrame({
    'Source': ['Yahoo Finance', 'FRED', 'EIA', 'Bloomberg', 'Quandl'],
    'Price Data': ['Excellent', 'Limited', 'Good (Energy)', 'Excellent', 'Excellent'],
    'Fundamentals': ['Poor', 'Good', 'Excellent (Energy)', 'Excellent', 'Good'],
    'Historical Depth': ['5-10 years', '20+ years', '10+ years', '30+ years', 'Varies'],
    'Update Frequency': ['Daily', 'Varies', 'Weekly/Monthly', 'Real-time', 'Varies'],
    'API Access': ['Yes (free)', 'Yes (free)', 'Yes (free)', 'Yes (paid)', 'Yes (paid)'],
    'Cost': ['Free', 'Free', 'Free', '$$$', '$$'],
    'Data Quality': ['Good', 'Excellent', 'Excellent', 'Excellent', 'Good-Excellent'],
})

print("Data Source Comparison:\n")
print(comparison.to_string(index=False))

## Part 2: API Integration

### 2.1 Using yfinance for Futures Prices

In [ ]:
# Initialize data fetcher
fetcher = CommodityDataFetcher(cache_dir='../../data/cache')

# Fetch crude oil futures data
print("Fetching WTI Crude Oil futures data...")
crude_data = fetcher.fetch_yahoo_data(
    ticker='CL=F',
    start_date='2020-01-01',
    end_date='2023-12-31'
)

print(f"\nData shape: {crude_data.shape}")
print(f"Columns: {list(crude_data.columns)}")
print(f"Date range: {crude_data.index.min().date()} to {crude_data.index.max().date()}")
crude_data.head()

In [ ]:
# Visualize price and volume
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Price
crude_data['Close'].plot(ax=axes[0], linewidth=1.5, color='blue')
axes[0].set_ylabel('Price ($/barrel)')
axes[0].set_title('WTI Crude Oil Futures - Continuous Contract')
axes[0].grid(True, alpha=0.3)

# Volume
crude_data['Volume'].plot(ax=axes[1], linewidth=1, color='green', alpha=0.7)
axes[1].set_ylabel('Volume')
axes[1].set_xlabel('Date')
axes[1].set_title('Trading Volume')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.2 Using FRED API for Economic Data

In [ ]:
# Fetch crude oil inventory data from FRED
print("Fetching crude oil inventory data from FRED...")
inventory_data = fetcher.fetch_fred_data(
    series_id='WCESTUS1',  # Weekly crude oil stocks
    start_date='2020-01-01',
    end_date='2023-12-31'
)

print(f"\nData shape: {inventory_data.shape}")
print(f"Date range: {inventory_data.index.min().date()} to {inventory_data.index.max().date()}")
inventory_data.head()

In [ ]:
# Visualize inventory levels
fig, ax = plt.subplots(figsize=(14, 6))

inventory_data.plot(ax=ax, linewidth=1.5, color='brown')

# Add 5-year average band
rolling_mean = inventory_data.rolling(window=52*5, center=True).mean()
rolling_std = inventory_data.rolling(window=52*5, center=True).std()

rolling_mean.plot(ax=ax, linewidth=2, linestyle='--', color='black', label='5-year average')
ax.fill_between(
    inventory_data.index,
    (rolling_mean - rolling_std).values.flatten(),
    (rolling_mean + rolling_std).values.flatten(),
    alpha=0.2,
    color='gray',
    label='±1 std dev'
)

ax.set_ylabel('Inventory (million barrels)')
ax.set_xlabel('Date')
ax.set_title('U.S. Crude Oil Inventory (Weekly)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.3 Fetching Multiple Series

In [ ]:
# Fetch multiple commodities
commodities = ['crude_oil', 'natural_gas', 'gold', 'copper']
multi_commodity_data = {}

print("Fetching multiple commodity prices...\n")
for commodity in commodities:
    ticker = COMMODITY_TICKERS[commodity]
    print(f"Fetching {commodity} ({ticker})...")
    
    data = fetcher.fetch_yahoo_data(
        ticker=ticker,
        start_date='2020-01-01',
        end_date='2023-12-31'
    )
    
    multi_commodity_data[commodity] = data['Close']

# Combine into single DataFrame
commodity_prices = pd.DataFrame(multi_commodity_data)
print(f"\nCombined data shape: {commodity_prices.shape}")
commodity_prices.head()

In [ ]:
# Normalize prices to 100 at start date for comparison
normalized_prices = (commodity_prices / commodity_prices.iloc[0] * 100)

fig, ax = plt.subplots(figsize=(14, 6))

for col in normalized_prices.columns:
    normalized_prices[col].plot(ax=ax, linewidth=2, label=col.replace('_', ' ').title())

ax.axhline(y=100, color='black', linestyle='--', alpha=0.5)
ax.set_ylabel('Normalized Price (Base = 100)')
ax.set_xlabel('Date')
ax.set_title('Relative Performance of Different Commodities')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.4 CommodityDataFetcher Class Demonstration

In [ ]:
# Use the high-level fetch method to get comprehensive dataset
comprehensive_data = fetcher.fetch_commodity_data(
    commodity='crude_oil',
    start_date='2020-01-01',
    end_date='2023-12-31',
    include_fundamentals=True,
    include_economic=True
)

print(f"Comprehensive dataset shape: {comprehensive_data.shape}")
print(f"\nColumns: {list(comprehensive_data.columns)}")
comprehensive_data.head()

In [ ]:
# Visualize relationship between price and inventory
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Price
ax1 = axes[0]
color = 'tab:blue'
ax1.set_ylabel('Price ($/barrel)', color=color)
ax1.plot(comprehensive_data.index, comprehensive_data['price'], color=color, linewidth=1.5)
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, alpha=0.3)
ax1.set_title('Price vs Inventory - Crude Oil')

# Inventory on secondary axis
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Inventory (million barrels)', color=color)
ax2.plot(comprehensive_data.index, comprehensive_data['inventory'], color=color, linewidth=1.5, alpha=0.7)
ax2.tick_params(axis='y', labelcolor=color)

# Scatter plot for correlation analysis
axes[1].scatter(
    comprehensive_data['inventory'],
    comprehensive_data['price'],
    alpha=0.5,
    s=20
)

# Add trend line
z = np.polyfit(
    comprehensive_data['inventory'].dropna(),
    comprehensive_data['price'].dropna(),
    1
)
p = np.poly1d(z)
axes[1].plot(
    comprehensive_data['inventory'].sort_values(),
    p(comprehensive_data['inventory'].sort_values()),
    "r--",
    linewidth=2,
    label=f'y={z[0]:.2f}x+{z[1]:.2f}'
)

# Calculate correlation
corr = comprehensive_data[['price', 'inventory']].corr().iloc[0, 1]
axes[1].set_xlabel('Inventory (million barrels)')
axes[1].set_ylabel('Price ($/barrel)')
axes[1].set_title(f'Price-Inventory Relationship (Correlation: {corr:.3f})')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 3: Data Quality Assessment

### 3.1 Completeness Checks

In [ ]:
def assess_completeness(df, threshold=0.95):
    """
    Assess data completeness for each column.
    
    Parameters
    ----------
    df : pd.DataFrame
        Dataset to assess
    threshold : float
        Minimum acceptable completeness (0-1)
    
    Returns
    -------
    pd.DataFrame
        Completeness statistics
    """
    results = []
    
    for col in df.columns:
        total = len(df)
        non_null = df[col].notna().sum()
        completeness = non_null / total
        
        results.append({
            'column': col,
            'total_rows': total,
            'non_null': non_null,
            'missing': total - non_null,
            'completeness': completeness,
            'meets_threshold': completeness >= threshold
        })
    
    return pd.DataFrame(results)

completeness = assess_completeness(comprehensive_data, threshold=0.95)
print("Data Completeness Assessment:\n")
print(completeness.to_string(index=False))

In [ ]:
# Visualize completeness
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['green' if x else 'red' for x in completeness['meets_threshold']]
completeness.plot(
    x='column',
    y='completeness',
    kind='bar',
    ax=ax,
    color=colors,
    legend=False
)

ax.axhline(y=0.95, color='orange', linestyle='--', linewidth=2, label='95% threshold')
ax.set_ylabel('Completeness Ratio')
ax.set_xlabel('Column')
ax.set_title('Data Completeness by Column')
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

### 3.2 Outlier Detection (Temporally Valid)

In [ ]:
def detect_outliers_expanding(series, threshold=3.0, min_periods=30):
    """
    Detect outliers using expanding window (temporally valid).
    
    At each point, only uses data available up to that point.
    """
    # Calculate expanding z-scores
    expanding_mean = series.expanding(min_periods=min_periods).mean()
    expanding_std = series.expanding(min_periods=min_periods).std()
    
    z_scores = (series - expanding_mean) / expanding_std
    
    # Flag outliers
    outliers = z_scores.abs() > threshold
    
    return outliers, z_scores

# Detect outliers in price returns
returns = comprehensive_data['price'].pct_change() * 100
outliers, z_scores = detect_outliers_expanding(returns, threshold=3.0)

print(f"Detected {outliers.sum()} outliers ({outliers.sum()/len(returns)*100:.2f}% of data)")
print(f"\nOutlier dates:")
print(returns[outliers].sort_values(ascending=False).head(10))

In [ ]:
# Visualize outliers
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Returns with outliers highlighted
returns.plot(ax=axes[0], linewidth=1, alpha=0.7, color='blue')
returns[outliers].plot(
    ax=axes[0],
    marker='o',
    linestyle='',
    markersize=8,
    color='red',
    label='Outliers'
)
axes[0].set_ylabel('Daily Return (%)')
axes[0].set_title('Price Returns with Outliers Highlighted')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='black', linestyle='-', alpha=0.3)

# Z-scores
z_scores.plot(ax=axes[1], linewidth=1, color='purple')
axes[1].axhline(y=3, color='red', linestyle='--', alpha=0.5, label='±3σ threshold')
axes[1].axhline(y=-3, color='red', linestyle='--', alpha=0.5)
axes[1].axhline(y=0, color='black', linestyle='-', alpha=0.3)
axes[1].fill_between(z_scores.index, -3, 3, alpha=0.1, color='green')
axes[1].set_ylabel('Z-Score')
axes[1].set_xlabel('Date')
axes[1].set_title('Expanding Window Z-Scores')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3.3 Cross-Source Validation

In [ ]:
# Compare WTI price from Yahoo Finance vs FRED
yahoo_wti = fetcher.fetch_yahoo_data(
    ticker='CL=F',
    start_date='2020-01-01',
    end_date='2023-12-31'
)['Close']

fred_wti = fetcher.fetch_fred_data(
    series_id='DCOILWTICO',  # WTI spot price
    start_date='2020-01-01',
    end_date='2023-12-31'
)

# Align data
comparison = pd.DataFrame({
    'yahoo_futures': yahoo_wti,
    'fred_spot': fred_wti
}).dropna()

# Calculate difference
comparison['difference'] = comparison['yahoo_futures'] - comparison['fred_spot']
comparison['pct_difference'] = (comparison['difference'] / comparison['fred_spot']) * 100

print("Cross-Source Validation Statistics:\n")
print(f"Correlation: {comparison['yahoo_futures'].corr(comparison['fred_spot']):.4f}")
print(f"Mean difference: ${comparison['difference'].mean():.2f}")
print(f"Mean % difference: {comparison['pct_difference'].mean():.2f}%")
print(f"Std of difference: ${comparison['difference'].std():.2f}")

In [ ]:
# Visualize cross-source comparison
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Both series
comparison[['yahoo_futures', 'fred_spot']].plot(ax=axes[0], linewidth=1.5, alpha=0.8)
axes[0].set_ylabel('Price ($/barrel)')
axes[0].set_title('WTI Price: Yahoo Finance Futures vs FRED Spot')
axes[0].legend(['Yahoo Futures', 'FRED Spot'])
axes[0].grid(True, alpha=0.3)

# Difference (futures premium)
comparison['difference'].plot(ax=axes[1], linewidth=1.5, color='green')
axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[1].fill_between(
    comparison.index,
    comparison['difference'],
    0,
    where=(comparison['difference'] > 0),
    alpha=0.3,
    color='green',
    label='Contango'
)
axes[1].fill_between(
    comparison.index,
    comparison['difference'],
    0,
    where=(comparison['difference'] < 0),
    alpha=0.3,
    color='red',
    label='Backwardation'
)
axes[1].set_ylabel('Futures Premium ($/barrel)')
axes[1].set_xlabel('Date')
axes[1].set_title('Futures-Spot Spread (Convenience Yield Signal)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 4: Handling Data Issues

### 4.1 Missing Data Imputation (Temporally Valid)

In [ ]:
def impute_missing_temporal(series, method='forward_fill', max_gap=5):
    """
    Impute missing values using temporally valid methods.
    
    Parameters
    ----------
    series : pd.Series
        Series with missing values
    method : str
        Imputation method:
        - 'forward_fill': Use last known value (most conservative)
        - 'linear': Linear interpolation (assumes smooth evolution)
        - 'expanding_mean': Use expanding window mean
    max_gap : int
        Maximum gap size to fill (larger gaps left as NaN)
    
    Returns
    -------
    pd.Series
        Imputed series
    """
    result = series.copy()
    
    if method == 'forward_fill':
        result = result.ffill(limit=max_gap)
    
    elif method == 'linear':
        # Only interpolate small gaps
        result = result.interpolate(method='linear', limit=max_gap)
    
    elif method == 'expanding_mean':
        # Fill with expanding mean up to that point
        expanding_mean = result.expanding(min_periods=1).mean()
        
        # Only fill small gaps
        mask = result.isna()
        gap_size = mask.groupby((~mask).cumsum()).cumsum()
        fill_mask = mask & (gap_size <= max_gap)
        
        result[fill_mask] = expanding_mean[fill_mask]
    
    return result

# Create sample series with missing data
sample = comprehensive_data['production'].copy()
sample_with_gaps = sample.copy()

# Introduce random gaps
np.random.seed(42)
gap_indices = np.random.choice(sample.index[50:], size=30, replace=False)
sample_with_gaps.loc[gap_indices] = np.nan

# Test different imputation methods
imputed_ff = impute_missing_temporal(sample_with_gaps, method='forward_fill', max_gap=5)
imputed_linear = impute_missing_temporal(sample_with_gaps, method='linear', max_gap=5)
imputed_mean = impute_missing_temporal(sample_with_gaps, method='expanding_mean', max_gap=5)

print(f"Original missing: {sample_with_gaps.isna().sum()}")
print(f"After forward fill: {imputed_ff.isna().sum()}")
print(f"After linear: {imputed_linear.isna().sum()}")
print(f"After expanding mean: {imputed_mean.isna().sum()}")

In [ ]:
# Visualize imputation methods
fig, ax = plt.subplots(figsize=(14, 6))

# Plot a zoomed-in section
zoom_range = slice('2022-01-01', '2022-06-30')

sample.loc[zoom_range].plot(ax=ax, linewidth=2, label='Original', color='black', alpha=0.7)
sample_with_gaps.loc[zoom_range].plot(
    ax=ax, marker='o', markersize=4, linestyle='', label='With Gaps', color='red'
)
imputed_ff.loc[zoom_range].plot(
    ax=ax, linewidth=1.5, linestyle='--', label='Forward Fill', alpha=0.7
)
imputed_linear.loc[zoom_range].plot(
    ax=ax, linewidth=1.5, linestyle='--', label='Linear', alpha=0.7
)
imputed_mean.loc[zoom_range].plot(
    ax=ax, linewidth=1.5, linestyle='--', label='Expanding Mean', alpha=0.7
)

ax.set_ylabel('Production')
ax.set_xlabel('Date')
ax.set_title('Comparison of Temporal Imputation Methods')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 4.2 Outlier Treatment

In [ ]:
def treat_outliers(series, method='winsorize', threshold=3.0, quantile=0.01):
    """
    Treat outliers using various methods.
    
    Parameters
    ----------
    series : pd.Series
        Series with potential outliers
    method : str
        Treatment method:
        - 'winsorize': Cap at threshold
        - 'remove': Set to NaN
        - 'median': Replace with expanding median
    threshold : float
        Z-score threshold for outlier detection
    quantile : float
        Quantile for winsorization
    
    Returns
    -------
    pd.Series
        Treated series
    """
    result = series.copy()
    
    # Detect outliers using expanding window
    expanding_mean = series.expanding(min_periods=30).mean()
    expanding_std = series.expanding(min_periods=30).std()
    z_scores = (series - expanding_mean) / expanding_std
    outliers = z_scores.abs() > threshold
    
    if method == 'winsorize':
        # Cap at expanding quantiles
        upper = series.expanding(min_periods=30).quantile(1 - quantile)
        lower = series.expanding(min_periods=30).quantile(quantile)
        result = result.clip(lower=lower, upper=upper)
    
    elif method == 'remove':
        result[outliers] = np.nan
    
    elif method == 'median':
        expanding_median = series.expanding(min_periods=30).median()
        result[outliers] = expanding_median[outliers]
    
    return result

# Treat outliers in returns
returns_winsorized = treat_outliers(returns, method='winsorize', threshold=3.0)
returns_removed = treat_outliers(returns, method='remove', threshold=3.0)
returns_median = treat_outliers(returns, method='median', threshold=3.0)

print("Outlier Treatment Comparison:\n")
print(f"Original - Mean: {returns.mean():.3f}%, Std: {returns.std():.3f}%")
print(f"Winsorized - Mean: {returns_winsorized.mean():.3f}%, Std: {returns_winsorized.std():.3f}%")
print(f"Removed - Mean: {returns_removed.mean():.3f}%, Std: {returns_removed.std():.3f}%")
print(f"Median - Mean: {returns_median.mean():.3f}%, Std: {returns_median.std():.3f}%")

### 4.3 Aligning Frequencies

In [ ]:
def align_frequencies(df, target_freq='D', method='forward_fill'):
    """
    Align mixed-frequency data to a common frequency.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with potentially mixed frequencies
    target_freq : str
        Target frequency ('D' for daily, 'W' for weekly, etc.)
    method : str
        How to handle missing values after resampling
    
    Returns
    -------
    pd.DataFrame
        Aligned DataFrame
    """
    # Resample to target frequency
    result = df.resample(target_freq).first()  # Take first value in period
    
    # Fill missing values
    if method == 'forward_fill':
        result = result.ffill()
    elif method == 'interpolate':
        result = result.interpolate(method='linear')
    
    return result

# Example: Align weekly inventory to daily price data
aligned_data = align_frequencies(
    comprehensive_data[['price', 'inventory']],
    target_freq='D',
    method='forward_fill'
)

print(f"Original data points: {len(comprehensive_data)}")
print(f"Aligned data points: {len(aligned_data)}")
print(f"\nMissing values after alignment:")
print(aligned_data.isna().sum())

In [ ]:
# Visualize frequency alignment
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Original (mixed frequency)
zoom = slice('2022-01-01', '2022-03-31')
comprehensive_data.loc[zoom, 'inventory'].plot(
    ax=axes[0], marker='o', linestyle='-', linewidth=1.5, markersize=6, label='Original (Weekly)'
)
axes[0].set_ylabel('Inventory (million barrels)')
axes[0].set_title('Original Weekly Inventory Data')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Aligned (daily)
aligned_data.loc[zoom, 'inventory'].plot(
    ax=axes[1], marker='o', linestyle='-', linewidth=1, markersize=3, label='Aligned (Daily, Forward Fill)'
)
axes[1].set_ylabel('Inventory (million barrels)')
axes[1].set_xlabel('Date')
axes[1].set_title('Aligned Daily Inventory Data')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 4.4 Building a Robust Data Pipeline

In [ ]:
def build_data_pipeline(commodity, start_date, end_date, 
                       quality_threshold=0.9, outlier_threshold=3.0):
    """
    Build a complete data pipeline with quality checks.
    
    Steps:
    1. Fetch data from multiple sources
    2. Assess quality
    3. Handle missing values
    4. Treat outliers
    5. Align frequencies
    6. Create point-in-time structure
    """
    print("=" * 60)
    print(f"Building data pipeline for {commodity}")
    print(f"Period: {start_date} to {end_date}")
    print("=" * 60)
    
    # Step 1: Fetch data
    print("\n1. Fetching data...")
    fetcher = CommodityDataFetcher(cache_dir='../../data/cache')
    data = fetcher.fetch_commodity_data(
        commodity=commodity,
        start_date=start_date,
        end_date=end_date,
        include_fundamentals=True,
        include_economic=True
    )
    print(f"   Fetched {len(data)} rows with {len(data.columns)} columns")
    
    # Step 2: Quality assessment
    print("\n2. Assessing quality...")
    completeness = assess_completeness(data, threshold=quality_threshold)
    failing_cols = completeness[~completeness['meets_threshold']]['column'].tolist()
    
    if failing_cols:
        print(f"   WARNING: {len(failing_cols)} columns below quality threshold: {failing_cols}")
    else:
        print("   All columns meet quality threshold")
    
    # Step 3: Handle missing values
    print("\n3. Handling missing values...")
    for col in data.columns:
        if data[col].isna().any():
            data[col] = impute_missing_temporal(data[col], method='forward_fill', max_gap=5)
    print(f"   Remaining missing values: {data.isna().sum().sum()}")
    
    # Step 4: Treat outliers
    print("\n4. Treating outliers...")
    for col in data.select_dtypes(include=[np.number]).columns:
        returns_col = data[col].pct_change()
        outliers, _ = detect_outliers_expanding(returns_col, threshold=outlier_threshold)
        n_outliers = outliers.sum()
        if n_outliers > 0:
            print(f"   {col}: {n_outliers} outliers detected and winsorized")
            data[col] = treat_outliers(data[col], method='winsorize', threshold=outlier_threshold)
    
    # Step 5: Align frequencies
    print("\n5. Aligning frequencies...")
    data = align_frequencies(data, target_freq='D', method='forward_fill')
    print(f"   Aligned to daily frequency: {len(data)} rows")
    
    # Step 6: Create point-in-time structure
    print("\n6. Creating point-in-time structure...")
    pit_data = PointInTimeDataFrame(data)
    print("   Point-in-time structure created")
    
    print("\n" + "=" * 60)
    print("Pipeline complete!")
    print("=" * 60)
    
    return pit_data, completeness

# Build pipeline for crude oil
clean_data, quality_report = build_data_pipeline(
    commodity='crude_oil',
    start_date='2020-01-01',
    end_date='2023-12-31',
    quality_threshold=0.9,
    outlier_threshold=3.0
)

In [ ]:
# Display final data quality
print("Final Data Quality Report:\n")
print(quality_report[['column', 'completeness', 'meets_threshold']].to_string(index=False))

print(f"\nFinal dataset shape: {clean_data.shape}")
print(f"Date range: {clean_data.index.min().date()} to {clean_data.index.max().date()}")
clean_data.head()

## Summary and Key Takeaways

### What We've Learned

1. **Data Sources**:
   - Free sources (Yahoo, FRED, EIA) provide substantial data for research
   - Premium sources offer more depth and reliability
   - Different sources have different strengths (prices vs fundamentals)

2. **API Integration**:
   - yfinance for futures prices
   - FRED for economic data and some commodity fundamentals
   - CommodityDataFetcher class for unified access

3. **Quality Assessment**:
   - Completeness: % of non-null values
   - Outlier detection: Using expanding windows to avoid look-ahead bias
   - Cross-source validation: Compare similar data from different sources

4. **Data Issues**:
   - Missing data: Use forward fill or interpolation (temporally valid)
   - Outliers: Winsorize or remove using expanding statistics
   - Mixed frequencies: Align to common frequency with forward fill

### Critical Principles

1. **Always use expanding windows** for statistics and outlier detection
2. **Never use future data** to impute or validate past data
3. **Document data provenance** - know where each data point came from
4. **Validate across sources** when possible
5. **Build automated pipelines** to ensure consistency

### Next Steps

In the next module, we'll use this clean, validated data to build fair value models:
- Inventory-based models
- Cost-of-carry models
- Supply-demand balance models
- Ensemble approaches

## Exercises

1. **Multi-Source Pipeline**: Build a data pipeline that fetches the same commodity from 3 different sources and creates a consensus estimate.

2. **Quality Monitoring**: Create a function that monitors data quality over time and alerts when quality degrades below thresholds.

3. **Advanced Outlier Detection**: Implement a more sophisticated outlier detector that considers:
   - Market regime (higher threshold in volatile periods)
   - Day-of-week effects
   - Seasonality

4. **Revision Tracking**: Build a system to track and store multiple vintages of the same data series.

5. **Custom Imputation**: Create a custom imputation method that uses related commodities to fill missing values (e.g., use Brent to estimate missing WTI values).